In [ ]:
import os
import math

from torch import Tensor
from typing import List, Dict
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer
from datasets import load_from_disk
from torch.utils.data import DataLoader

In [1]:
def single_head_attention(X, Wk, Wq, Wv, need_mask=False)->Tensor:
    """
    单头注意力计算
    :param X: 基础输入矩阵, n*dim
    :param Wk: key参数矩阵, n*dim
    :param Wq: query参数矩阵, n*dim
    :param Wv: value参数矩阵, n*dim
    :param need_mask: 是否需要进行因果mask
    """
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    # relevance: n*n
    relevance = torch.matmul(Q, K.T)/Q.shape[-1]**0.5
    # Q 我需要关注什么 K 我可以提供哪些信息
    # rij = qi·kj, 在需要因果mask时, 应使qi看不到i之后的k, 即不含对角线的上三角mask
    if need_mask:
        # torch.triu(): 构建上三角True方阵
        # diagonal: 主对角线为0，其值表示某条对角线及以上为True
        mask = torch.triu(torch.ones(relevance.shape[1], relevance.shape[1]), diagonal=1)
        relevance = relevance.masked_fill(mask, float('-inf'))
    relevance = torch.softmax(relevance, dim=-1)
    att = relevance @ V
    return att

NameError: name 'Tensor' is not defined

In [7]:
def multi_head_attention(x_q, x_k, x_v, w_q, w_k, w_v, w_o, head, need_mask=False)->Tensor:
    """
    多头注意力计算
    :param x_q: [batch, n, dim]
    :param x_k: [batch, n, dim]
    :param x_v: [batch, n, dim]
    :param w_q: [dim, dim]
    :param w_k: [dim, dim]
    :param w_v: [dim, dim]
    :param w_o: 多头汇聚参数矩阵, [dim, dim]
    :param head: 多头注意力头数
    :param need_mask: need_mask: 是否需要进行因果mask
    """
    dim = x_q.shape[-1]
    assert dim % head == 0
    # q/k/v: [batch, n, dim]
    q = x_q @ w_q
    k = x_k @ w_k
    v = x_v @ w_v

    # q/k/v: [batch, head, n, dim//head]
    q = q.reshape(q.shape[0], q.shape[1], head, -1).transpose(1, 2)
    k = k.reshape(k.shape[0], k.shape[1], head, -1).transpose(1, 2)
    v = v.reshape(v.shape[0], v.shape[1], head, -1).transpose(1, 2)

    # relevance: [batch, head, n, n]
    relevance = torch.matmul(q, k.transpose(-2, -1))/(dim//head)**0.5
    if need_mask:
        # mask设备与relevance对齐
        mask = torch.triu(torch.ones(relevance.shape[-1], relevance.shape[-1]), diagonal=1).to(relevance.device)
        relevance = relevance.masked_fill(mask, float('-inf'))
    relevance = torch.softmax(relevance, dim=-1)

    # att: [batch, head, n, dim//head]
    att = relevance @ v
    # att: [batch, n, dim]
    att = att.transpose(1, 2).contiguous()
    att = att.reshape(att.shape[0], att.shape[1], -1)
    att = att @ w_o
    return att

In [8]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim:int, head:int, dropout:float=0.1, need_mask=False):
        super().__init__()
        self.w_q = nn.Parameter(torch.normal(mean=0, std=0.01, size=(dim, dim)), requires_grad=True)
        self.w_k = nn.Parameter(torch.normal(mean=0, std=0.01, size=(dim, dim)), requires_grad=True)
        self.w_v = nn.Parameter(torch.normal(mean=0, std=0.01, size=(dim, dim)), requires_grad=True)
        self.w_o = nn.Parameter(torch.normal(mean=0, std=0.01, size=(dim, dim)), requires_grad=True)
        self.h = head
        self.mask = need_mask

        self.dropout = nn.Dropout(dropout)

    def forward(self, x_q:Tensor, x_k:Tensor, x_v:Tensor, pad_mask:Tensor=None):
        dim = x_q.shape[-1]

        assert dim % self.h == 0

        # q/k/v: [batch, n, dim]
        q = x_q @ self.w_q
        k = x_k @ self.w_k
        v = x_v @ self.w_v

        # q/k/v: [batch, head, n, dim//head]
        q = q.reshape(q.shape[0], q.shape[1], self.h, -1).transpose(1, 2).contiguous()
        k = k.reshape(k.shape[0], k.shape[1], self.h, -1).transpose(1, 2).contiguous()
        v = v.reshape(v.shape[0], v.shape[1], self.h, -1).transpose(1, 2).contiguous()

        # pad_mask: [batch, n] ---.unsqueeze(1).unsqueeze(2)---> [batch, 1, 1, n]
        # relevance: [batch, head, n, n]
        k_t = k.transpose(-2, -1).contiguous()
        relevance = torch.matmul(q, k_t)/(dim//self.h)**0.5

        if pad_mask is not None:
            relevance = relevance.masked_fill(pad_mask.unsqueeze(1).unsqueeze(2), float('-inf'))
        if self.mask:
            mask = torch.triu(torch.ones(relevance.shape[-1], relevance.shape[-1], dtype=torch.bool), diagonal=1).to(relevance.device)
            relevance = relevance.masked_fill(mask, float('-inf'))
        relevance = torch.softmax(relevance, dim=-1)

        relevance = self.dropout(relevance)

        # att: [batch, head, n, dim//head]
        att = relevance @ v
        # att: [batch, n, dim]
        att = att.transpose(1, 2).contiguous()
        att = att.reshape(att.shape[0], att.shape[1], -1)
        att = att @ self.w_o
        return att

In [9]:
class FFN(nn.Module):
    """
    Transformer内置双层全连接前向层
    :param dim: 输入维度
    :param d_ff: 全连接层扩张维度，默认4倍扩张
    :param dropout: 丢弃率
    """
    def __init__(self, dim:int, d_ff:int=None, dropout:float=0.1):
        super().__init__()
        if d_ff is None:
            d_ff = dim * 4

        self.layer1 = nn.Linear(dim, d_ff)
        self.layer2 = nn.Linear(d_ff, dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
    def forward(self, x:Tensor):
        x = self.relu(self.layer1(x))
        return self.layer2(self.dropout(x))

In [10]:
class LN(nn.Module):
    def __init__(self, dim, eps:float=1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
    def forward(self, x:Tensor):
        """
        :param x: [batch, n, dim]
        """
        mean = x.mean(dim=-1, keepdim=True)
        # unbiased: 是否使用无偏估计(是: N-1, 否: N)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x + self.beta

In [11]:
class EncoderBlock(nn.Module):
    """
    post_layer_norm的transformer encoder块
    """
    def __init__(self, dim:int, head:int, eps:float=1e-5, d_ff:int=None, dropout:float=0.1):
        super().__init__()
        self.h = head

        self.attention = MultiHeadAttention(dim, self.h)
        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = LN(dim, eps)

        self.ffn = FFN(dim, d_ff, dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = LN(dim, eps)
    def forward(self, x:Tensor, pad_mask:Tensor):
        x = x + self.dropout1(self.attention(x, x, x, pad_mask))
        x = self.ln1(x)

        x = x + self.dropout2(self.ffn(x))
        x = self.ln2(x)
        return x

class Encoder(nn.Module):
    def __init__(self, n:int, dim:int, head:int, eps:float=1e-5, d_ff:int=None, dropout:float=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderBlock(dim, head, eps, d_ff, dropout) for _ in range(n)]
        )
    def forward(self, x:Tensor, pad_mask:Tensor):
        for encoder in self.layers:
            x = encoder(x, pad_mask)
        return x

In [12]:
class DecoderBlock(nn.Module):
    def __init__(self, dim:int, head:int, eps=1e-5, d_ff:int=None, dropout:float=0.1):
        super().__init__()

        self.h = head

        self.mask_attention = MultiHeadAttention(dim, self.h, need_mask=True)
        self.dropout1 = nn.Dropout(dropout)
        self.ln1 = LN(dim, eps)

        self.attention = MultiHeadAttention(dim, self.h)
        self.dropout2 = nn.Dropout(dropout)
        self.ln2 = LN(dim, eps)

        self.ffn = FFN(dim, d_ff, dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.ln3 = LN(dim, eps)
    def forward(self, x:Tensor, encoder_output:Tensor, src_pad_mask:Tensor, tgt_pad_mask:Tensor):
        x = x + self.dropout1(self.mask_attention(x, x, x, tgt_pad_mask))
        x = self.ln1(x)

        x = x + self.dropout2(self.attention(x, encoder_output, encoder_output, src_pad_mask))
        x = self.ln2(x)

        x = x + self.dropout3(self.ffn(x))
        x = self.ln3(x)
        return x

class Decoder(nn.Module):
    def __init__(self, n:int, dim:int, head:int, eps:float=1e-5, d_ff:int=None, dropout:float=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderBlock(dim, head, eps, d_ff, dropout) for _ in range(n)]
        )
    def forward(self, x:Tensor, encoder_output:Tensor, src_pad_mask:Tensor, tgt_pad_mask:Tensor):
        for decoder in self.layers:
            x = decoder(x, encoder_output, src_pad_mask, tgt_pad_mask)
        return x

In [13]:
class LinearBlock(nn.Module):
    def __init__(self, dim:int, emb_dim:int):
        super().__init__()
        self.layer = nn.Linear(dim, emb_dim)
    def forward(self, x:Tensor):
        return self.layer(x)

In [14]:
class Transformer(nn.Module):
    def __init__(self, layer_num:int, d_model:int, src_emb_sz:int, tgt_emb_sz:int, head:int, eps:float=1e-5, d_ff:int=None, dropout:float=0.1):
        super().__init__()
        self.d_model = d_model

        self.src_emb = nn.Embedding(src_emb_sz, self.d_model)
        self.tgt_emb = nn.Embedding(tgt_emb_sz, self.d_model)

        self.encoder = Encoder(layer_num, self.d_model, head, eps, d_ff, dropout)
        self.decoder = Decoder(layer_num, self.d_model, head, eps, d_ff, dropout)
        self.linear = LinearBlock(self.d_model, tgt_emb_sz)

    def positional_encoding(self, n:int):
        pe = torch.zeros(n, self.d_model)
        meta = torch.exp(torch.arange(0, self.d_model, 2) * -math.log(10000) / self.d_model)
        # position = torch.arange(0, n).unsqueeze(1)
        position = torch.arange(0, n).unsqueeze(1)
        pe[:, 0::2] = torch.sin(position * meta)
        pe[:, 1::2] = torch.cos(position * meta)
        return pe

    def encode(self, src_ids:Tensor, src_pad_mask:Tensor):
        src = self.src_emb(src_ids) * math.sqrt(self.d_model)
        pe_src = self.positional_encoding(src.shape[-2])
        src = src + pe_src.to(src.device)
        return self.encoder(src, src_pad_mask)

    def decode(self, tgt_ids:Tensor, tgt_pad_mask:Tensor, encoder_output:Tensor, src_pad_mask:Tensor):
        tgt = self.tgt_emb(tgt_ids) * math.sqrt(self.d_model)
        pe_tgt = self.positional_encoding(tgt.shape[-2])
        tgt = tgt + pe_tgt.to(tgt.device)
        return self.decoder(tgt, encoder_output, src_pad_mask, tgt_pad_mask)

    def forward(self, src_ids:Tensor, tgt_ids:Tensor, src_pad_mask:Tensor, tgt_pad_mask:Tensor):
        encoder_output = self.encode(src_ids, src_pad_mask)
        tgt = self.decode(tgt_ids, tgt_pad_mask, encoder_output)
        tgt = self.linear(tgt)
        return tgt

In [15]:
class TransformerLRScheduler:
    def __init__(self, optimizer, d_model, warmup_steps):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup = warmup_steps
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.d_model**(-0.5) * min(
            self.step_num**(-0.5),                          # 衰减阶段
            self.step_num * self.warmup**(-1.5)              # 预热阶段
        )
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

In [16]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
vocab_sz = tokenizer.vocab_size

LAYER_NUM = 6
D_MODEL = 512
SRC_EMB_SZ = vocab_sz
TGT_EMB_SZ = vocab_sz
HEAD = 8
EPS = 1e-5
D_FF = None
DROPOUT = 0.1
MAX_LENGTH = 128
beta1 = 0.9; beta2 = 0.98
l_eps = 1e-9
warmup = 4000
EPOCHS = 1

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tf = Transformer(layer_num=LAYER_NUM, d_model=D_MODEL, src_emb_sz=SRC_EMB_SZ, tgt_emb_sz=TGT_EMB_SZ, head=HEAD, eps=EPS, d_ff=D_FF, dropout=DROPOUT).to(device)
optimizer = torch.optim.Adam(params=tf.parameters(),betas=(beta1, beta2),eps=l_eps)
lr_schedular = TransformerLRScheduler(optimizer = optimizer, d_model=D_MODEL, warmup_steps=warmup)
criterion = torch.nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

In [17]:
data_path = "D:/Learn/machine_learning/Transformer/data/wikitext2"

# WikiText-2（原始文本版本，保留原始词汇）
# dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
# dataset.save_to_disk(data_path)

# 过滤空、极短文本
train_data = load_from_disk(os.path.join(data_path, "train"))
# filter: 函数返回值为False的元素删除
train_data = train_data.filter(lambda x: len(x['text'].strip()) > 10)

def tokenize(tokenizer, batch)->Dict[str, List]:
    return {'input_ids': [tokenizer.encode(x)[:MAX_LENGTH] for x in batch]}
# map: 对元素做形式、内容转换
train_data = train_data.map(lambda x: tokenize(tokenizer, x['text']), batched=True, remove_columns=['text'])

# 复制任务：[input_ids, label]
def add_labels(batch):
    return {'labels': batch['input_ids']}
train_data = train_data.map(lambda x: add_labels(x), batched=True)

def collate_fn(batch):
    input_ids = [torch.tensor(x['input_ids']) for x in batch]
    labels = [torch.tensor(x['labels']) for x in batch]

    src = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    tgt = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=tokenizer.pad_token_id)

    src_pad_mask = (src == tokenizer.pad_token_id)
    # 复制任务下tgt_pad_mask=src_pad_mask
    tgt_pad_mask = src_pad_mask

    return src, tgt, src_pad_mask, tgt_pad_mask

train_loader = DataLoader(train_data, collate_fn=collate_fn, batch_size=8, shuffle=True)

In [14]:
for epoch in range(EPOCHS):
    epoch_loss = 0
    cnt = 0
    for batch in train_loader:
        print(f"Batch No: {cnt + 1}")
        src, tgt, src_pad_mask, tgt_pad_mask = batch
        src = src.to(device)
        tgt = tgt.to(device)
        src_pad_mask = src_pad_mask.to(device)
        tgt_pad_mask = tgt_pad_mask.to(device)

        optimizer.zero_grad()
        y = tf(src, tgt[:, :-1], src_pad_mask, tgt_pad_mask[:, :-1])
        loss = criterion(y.reshape(-1,vocab_sz), tgt[: , 1:].reshape(-1))

        epoch_loss += loss.item()
        cnt+=1

        # torch.autograd.set_detect_anomaly(True)
        loss.backward()
        optimizer.step()
        lr_schedular.step()
    print("Epoch: ", epoch, "; Average loss: ", epoch_loss/cnt)
torch.save(tf.state_dict(), "transformer.pt")

Batch No: 1
Batch No: 2
Batch No: 3
Batch No: 4
Batch No: 5
Batch No: 6
Batch No: 7
Batch No: 8
Batch No: 9
Batch No: 10
Batch No: 11
Batch No: 12
Batch No: 13
Batch No: 14
Batch No: 15
Batch No: 16
Batch No: 17
Batch No: 18
Batch No: 19
Batch No: 20
Batch No: 21
Batch No: 22
Batch No: 23
Batch No: 24
Batch No: 25
Batch No: 26
Batch No: 27
Batch No: 28
Batch No: 29
Batch No: 30
Batch No: 31
Batch No: 32
Batch No: 33
Batch No: 34
Batch No: 35
Batch No: 36
Batch No: 37
Batch No: 38
Batch No: 39
Batch No: 40
Batch No: 41
Batch No: 42
Batch No: 43
Batch No: 44
Batch No: 45
Batch No: 46
Batch No: 47
Batch No: 48
Batch No: 49
Batch No: 50
Batch No: 51
Batch No: 52
Batch No: 53
Batch No: 54
Batch No: 55
Batch No: 56
Batch No: 57
Batch No: 58
Batch No: 59
Batch No: 60
Batch No: 61
Batch No: 62
Batch No: 63
Batch No: 64
Batch No: 65
Batch No: 66
Batch No: 67
Batch No: 68
Batch No: 69
Batch No: 70
Batch No: 71
Batch No: 72
Batch No: 73
Batch No: 74
Batch No: 75
Batch No: 76
Batch No: 77
Batch No

In [15]:
tf = Transformer(layer_num=LAYER_NUM, d_model=D_MODEL, src_emb_sz=SRC_EMB_SZ, tgt_emb_sz=TGT_EMB_SZ, head=HEAD, eps=EPS, d_ff=D_FF, dropout=DROPOUT).to(device)
tf.load_state_dict(torch.load("transformer.pt", map_location=device))
# 过滤空、极短文本
test_data = load_from_disk(os.path.join(data_path, "test"))
# filter: 函数返回值为False的元素删除
test_data = test_data.filter(lambda x: len(x['text'].strip()) > 10)
# map: 对元素做形式、内容转换
test_data = test_data.map(lambda x: tokenize(tokenizer, x['text']), batched=True, remove_columns=['text'])
# 复制任务：[input_ids, label]
test_data = test_data.map(lambda x: add_labels(x), batched=True)
test_loader = DataLoader(test_data, collate_fn=collate_fn, batch_size=8, shuffle=False)

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

In [16]:
test_loss = 0
cnt = 0
tf.eval()
with torch.no_grad():
    for batch in test_loader:
        print(f"Batch No: {cnt + 1}")
        src, tgt, src_pad_mask, tgt_pad_mask = batch
        src = src.to(device)
        tgt = tgt.to(device)
        src_pad_mask = src_pad_mask.to(device)
        tgt_pad_mask = tgt_pad_mask.to(device)

        y = tf(src, tgt[:, :-1], src_pad_mask, tgt_pad_mask[:, :-1])
        loss = criterion(y.reshape(-1,vocab_sz), tgt[: , 1:].reshape(-1))
        test_loss += loss.item()
        cnt += 1
    print("Average loss: ", test_loss/cnt)

Batch No: 1
Batch No: 2
Batch No: 3
Batch No: 4
Batch No: 5
Batch No: 6
Batch No: 7
Batch No: 8
Batch No: 9
Batch No: 10
Batch No: 11
Batch No: 12
Batch No: 13
Batch No: 14
Batch No: 15
Batch No: 16
Batch No: 17
Batch No: 18
Batch No: 19
Batch No: 20
Batch No: 21
Batch No: 22
Batch No: 23
Batch No: 24
Batch No: 25
Batch No: 26
Batch No: 27
Batch No: 28
Batch No: 29
Batch No: 30
Batch No: 31
Batch No: 32
Batch No: 33
Batch No: 34
Batch No: 35
Batch No: 36
Batch No: 37
Batch No: 38
Batch No: 39
Batch No: 40
Batch No: 41
Batch No: 42
Batch No: 43
Batch No: 44
Batch No: 45
Batch No: 46
Batch No: 47
Batch No: 48
Batch No: 49
Batch No: 50
Batch No: 51
Batch No: 52
Batch No: 53
Batch No: 54
Batch No: 55
Batch No: 56
Batch No: 57
Batch No: 58
Batch No: 59
Batch No: 60
Batch No: 61
Batch No: 62
Batch No: 63
Batch No: 64
Batch No: 65
Batch No: 66
Batch No: 67
Batch No: 68
Batch No: 69
Batch No: 70
Batch No: 71
Batch No: 72
Batch No: 73
Batch No: 74
Batch No: 75
Batch No: 76
Batch No: 77
Batch No

In [21]:
@torch.no_grad()
def generate(model, tokenizer, input_text:str, max_length:int, device='cuda'):
    """
    输入文本，模型复制输出
    返回: (输入字符串, 输出字符串, token级准确率)
    """
    model.eval()
    # 1. Tokenize 输入
    input_ids = tokenizer.encode(input_text)[:max_length]
    input_len = len(input_ids)
    src = torch.tensor([input_ids]).to(device)
    src_pad_mask = (src == tokenizer.pad_token_id)
    # 2. Encoder — 只跑一次
    encoder_output = model.encode(src, src_pad_mask)
    # 3. Decoder — 自回归生成
    # 起始 token: eos_token（在复制任务中充当 BOS）
    generated = [tokenizer.eos_token_id]
    for step in range(max_length - 1):
        tgt = torch.tensor(generated).unsqueeze(0).to(device)
        tgt_pad_mask = torch.zeros_like(tgt, dtype=torch.bool)
        # 取最后一个位置的 logits
        logits = model.decode(tgt, encoder_output, src_pad_mask, tgt_pad_mask)
        next_token_logits = logits[:, -1, :]  # [1, vocab_sz]
        # 贪心解码
        next_token = next_token_logits.argmax(dim=-1).item()
        generated.append(next_token)
        if next_token == tokenizer.eos_token_id:
            break
    # 4. Detokenize：跳过起始 eos
    output_ids = generated[1:]
    output_text = tokenizer.decode(output_ids, skip_special_tokens=True)
    # 5. 评估复制准确率
    min_len = min(input_len, len(output_ids))
    correct = sum(1 for i in range(min_len) if input_ids[i] == output_ids[i])
    accuracy = correct / max(input_len, len(output_ids))
    return input_text, output_text, accuracy

In [25]:
tf = Transformer(layer_num=LAYER_NUM, d_model=D_MODEL, src_emb_sz=SRC_EMB_SZ, tgt_emb_sz=TGT_EMB_SZ, head=HEAD, eps=EPS, d_ff=D_FF, dropout=DROPOUT).to(device)
tf.load_state_dict(torch.load("transformer.pt", map_location=device))
input = "Chance fights ever on the side of the prudent."
print(generate(tf, tokenizer, input, MAX_LENGTH, device='cuda'))

AssertionError: Torch not compiled with CUDA enabled